# Hi-EF Phase 1 validation interventions

Evaluate five frozen full-model checkpoints under fixed clip-III interventions. This notebook performs no training and never evaluates the test partition. Attach both `ptrnghieu/hi-ef-features-v2` and the output of `ptrnghieu/hief-multiseed-validation`.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/phase1_interventions')

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)

assert (FEATURES / '01_00059.pt').exists(), 'Attach ptrnghieu/hi-ef-features-v2'
print('Repository and features ready')

In [ ]:
# Locate the attached canonical multi-seed notebook output by content,
# rather than relying on a Kaggle-generated directory name.
candidates = list(Path('/kaggle/input').rglob('validation_matrix_summary.json'))
valid = []
for summary_path in candidates:
    root = summary_path.parent
    if all((root / f'full_seed{seed}' / 'best.pt').exists() for seed in (42, 123, 456, 789, 1024)):
        valid.append(root)

if len(valid) != 1:
    raise RuntimeError(
        'Expected exactly one attached hief-multiseed-validation output; '
        f'found {len(valid)} valid roots: {valid}'
    )
CHECKPOINTS = valid[0]
print('Checkpoints:', CHECKPOINTS)

In [ ]:
command = [
    'python', str(REPO / 'experiments/run_validation_interventions.py'),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--features-dir', str(FEATURES),
    '--checkpoints-dir', str(CHECKPOINTS),
    '--output-dir', str(OUTPUT),
    '--seeds', '42', '123', '456', '789', '1024',
    '--replacement-seed-start', '1701',
    '--replacement-replicates', '20',
    '--batch-size', '32', '--workers', '2'
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd

summary = json.loads((OUTPUT / 'intervention_summary.json').read_text())
assert summary['test_evaluated'] is False
display(pd.read_csv(OUTPUT / 'intervention_results.csv').head(20))
print(json.dumps(summary['aggregate'], indent=2))

Run via **Save Version → Save & Run All**. Preserve the entire `/kaggle/working/phase1_interventions` directory. The principal outputs are `intervention_summary.json`, `intervention_results.csv`, and the fixed replacement manifests.